**News Sentiment as a Trading Signal: Measuring Predictive Decay Across Holding Horizons**

Syed Sirajuddin · Master of Science in Applied Artificial Intelligence · Shiley Marcos School of Engineering, University of San Diego · AAI-590 Capstone

# Notebook 1 of 5 — Introduction, Data Acquisition, and Cleaning

This notebook series is organized to mirror both the required elements of the capstone code base and the sections of the final report. The mapping is as follows, so that a reviewer can locate each required element directly:

| Notebook | Code-base requirement | Report section(s) |
| --- | --- | --- |
| 01 — this notebook | Data Cleaning | Introduction; Data Summary |
| 02 — Exploratory Data Analysis | Exploratory Data Analysis | Data Summary |
| 03 — Sentiment Model Fine-Tuning | Model Training | Literature Review; Methodology |
| 04 — Signals & Multi-Horizon Backtesting | Model/Pipeline Design and Building | Methodology |
| 05 — Optimization & Analysis | Model Optimization; Analysis and Discussion | Methodology; Results; Conclusion |

Reusable logic lives in the `src/` package and is imported here rather than duplicated, so that the notebooks remain readable while the repository retains a single, tested implementation of each component. Results-oriented commentary is deferred: The interpretation and summary cells are completed from the executed run.


## 1. Introduction

Modern equity markets generate a continuous, high-volume stream of price-relevant news — earnings releases, analyst rating changes, product announcements, and regulatory actions — far more than any individual investor can read and act on in a timely way. Under the efficient market hypothesis, public information should be incorporated into prices almost immediately (Fama, 1970); empirically, however, a substantial literature documents that the textual tone of news predicts returns with a measurable lag. Tetlock (2007) showed that media pessimism predicts downward pressure on prices followed by reversion, and Heston and Sinha (2017) found that the horizon of predictability depends strongly on how sentiment is aggregated, with daily news sentiment predicting returns over one to two days but weekly aggregation extending predictability to a quarter.

This project asks a deliberately comparative question: **if daily financial-news sentiment carries tradable information, over what holding horizon does that information remain exploitable?** Rather than committing to a single trading style, we treat the holding horizon as the primary experimental variable and evaluate the *same* sentiment-derived signals at holds ranging from one day to six months, against buy-and-hold and no-skill baselines. Our hypothesis, motivated by the reversal and decay patterns in the literature cited above, is that predictive value is strongest at short (swing) horizons and decays as the holding period lengthens. A finding that the signal disappears beyond a certain horizon is as informative as a finding that it persists.

The intended end user is a retail investor who actively manages a portfolio and needs a research and decision-support screener — not a black-box autopilot, and not a high-frequency system. In a deployed setting, the pipeline built here would consume streaming news and live market data; for research and backtesting, we reconstruct both streams historically on a strict point-in-time basis.

Two data sources feed the project: (1) daily open-high-low-close-volume (OHLCV) bars for a universe of twenty liquid, large-capitalization U.S. equities, obtained through the open-source `yfinance` library (Aroussi, 2023); and (2) historical financial-news headlines linked to those tickers, obtained from the Alpha Vantage market-news-and-sentiment API, whose archive extends back to March 2022 and which supplies its own per-article sentiment score alongside each headline. A third, auxiliary dataset — the Financial PhraseBank of roughly 4,800 expert-annotated sentences (Malo, Sinha, Korhonen, Wallenius, & Takala, 2014) — is used in Notebook 03 to fine-tune and validate the sentiment model.


In [1]:
#!python -m pip install -r requirements.txt

In [2]:
#!python -m pip uninstall -y torch
#!python -m pip install torch --index-url https://download.pytorch.org/whl/cu124 # Only for Windows with CUDA 12.4
#!python -m pip install python-dotenv -q

In [3]:
%matplotlib inline
# Environment setup: resolve the repository root so `src` imports work
# whether this notebook is run from notebooks/ or the project root.
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

print(f"Project root: {ROOT}")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 40)
RANDOM_SEED = 42

Project root: d:\assignments\Assignments\AAI590\AAI590FinalProject


In [4]:
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")

## 2. Price Data: Universe, Acquisition, and Cleaning

The universe consists of twenty large-cap U.S. names spanning technology, financials, health care, energy, consumer, and industrial sectors. Two considerations drive this choice. First, news-to-ticker linkage — the noisiest step in any news-based pipeline — is far more reliable for heavily covered large caps than for small caps, where sparse and ambiguous coverage would contaminate the signal. Second, high liquidity keeps the simple transaction-cost model used in backtesting (a fixed per-side charge) defensible; for illiquid names, market impact would dominate and require a more elaborate cost model.

The date range (January 2020 through December 2025) covers several distinct market regimes — the 2020 pandemic crash and recovery, the 2022 drawdown, and subsequent recoveries — which matters because a sentiment signal that only works in one regime is of limited practical value.


In [5]:
from src.config import UNIVERSE
from src.data.prices import download_prices, clean_prices

print(f"Universe ({len(UNIVERSE.tickers)} tickers): {', '.join(UNIVERSE.tickers)}")
print(f"Date range: {UNIVERSE.start_date} to {UNIVERSE.end_date}")

# Download raw OHLCV bars (writes data/raw/prices_raw.parquet)
prices_raw = download_prices()
prices_raw.head()

Universe (20 tickers): AAPL, MSFT, GOOGL, AMZN, META, NVDA, TSLA, JPM, JNJ, XOM, WMT, PG, V, UNH, HD, DIS, NFLX, AMD, BA, PFE
Date range: 2020-01-01 to 2025-12-31


2026-08-09 20:44:02,817 Downloaded 30140 rows for 20 tickers


Price,date,open,high,low,close,adj_close,volume,ticker
0,2020-01-02,74.059998,75.150002,73.797501,75.087502,72.333878,135480400,AAPL
1,2020-01-03,74.287498,75.144997,74.125000,74.357498,71.630669,146322800,AAPL
2,2020-01-06,73.447502,74.989998,73.187500,74.949997,72.201408,118387200,AAPL
3,2020-01-07,74.959999,75.224998,74.370003,74.597504,71.861832,108872000,AAPL
4,2020-01-08,74.290001,76.110001,74.290001,75.797501,73.017822,132079200,AAPL


### 2.1 Cleaning rules and their rationale

Four cleaning rules are applied, each targeting a data-quality issue with a known source. In a fully productionized pipeline, each of these checks would run as an automated data-validation stage (with alerting) rather than as a one-time cleaning step:

1. **Missing OHLC values** are dropped. These are rare and typically correspond to trading halts or vendor gaps; imputation would fabricate prices on days when no trade was possible.
2. **Single-day volume gaps** are forward-filled; longer gaps are dropped, since extended missing volume usually indicates a data-vendor problem rather than a market event.
3. **Insufficient history** — tickers with fewer than one year of observations are excluded, because the longest holding horizon studied (126 trading days) requires substantial history for even a handful of non-overlapping trades.
4. **Extreme single-day returns** (beyond ±50%) that are *not* corroborated by a volume spike are winsorized. Genuine 50% moves are accompanied by heavy trading; a "quiet" extreme move is almost always a bad print or an unadjusted split artifact from the vendor.

All return computations downstream use the **split- and dividend-adjusted close**, so that corporate actions do not masquerade as price moves.


In [6]:
prices = clean_prices(prices_raw)
print(f"Clean panel: {len(prices):,} rows, "
      f"{prices['ticker'].nunique()} tickers, "
      f"{prices['date'].min().date()} to {prices['date'].max().date()}")
prices.groupby("ticker")["date"].agg(["min", "max", "count"])

2026-08-09 20:44:02,833 Dropped 0 rows with missing OHLC


Clean panel: 30,140 rows, 20 tickers, 2020-01-02 to 2025-12-30


,min,max,count
ticker,,,
AAPL,2020-01-02,2025-12-30,1507
AMD,2020-01-02,2025-12-30,1507
AMZN,2020-01-02,2025-12-30,1507
BA,2020-01-02,2025-12-30,1507
DIS,2020-01-02,2025-12-30,1507
GOOGL,2020-01-02,2025-12-30,1507
HD,2020-01-02,2025-12-30,1507
JNJ,2020-01-02,2025-12-30,1507
JPM,2020-01-02,2025-12-30,1507


## 3. News Data: Acquisition, Deduplication, and Point-in-Time Alignment

News is drawn from the Alpha Vantage market-news-and-sentiment interface, a documented public API whose archive begins in March 2022 and which returns, for each article, a ticker-level relevance score and its own sentiment score alongside topic classifications. Each article is normalized to a common schema (publication timestamp, ticker, headline, summary, source, and URL), with two extra columns carrying Alpha Vantage's own sentiment score and relevance so that Notebook 06 can benchmark it against the project's FinBERT model.

Two cleaning steps follow. **Deduplication** removes syndicated wire copies — the same headline republished by many outlets — which would otherwise let a single news event masquerade as many independent signals and overweight it in the daily aggregate. **Length filtering** drops fragments under roughly ten characters, which are almost always feed artifacts rather than headlines.

### 3.1 The point-in-time guarantee

The single most important design decision in this notebook is the mapping of every article to its **effective date**: the first trading session on which it could have been acted upon. An article published after the market close, on a weekend, or on a holiday is rolled forward to the next session. This guards against look-ahead bias — using information in a backtest that was not yet available at the time — which is among the most common ways a strategy backtest overstates performance (López de Prado, 2018). Because the mapping is applied once, here at ingestion, every downstream stage inherits it automatically.

In [7]:
import os
from src.config import RAW_DIR, PROCESSED_DIR, UNIVERSE
from src.data.news_alphavantage import (fetch_alphavantage, completeness_report,
                                         ARCHIVE_START)
from src.data.news import clean_news

# Alpha Vantage key (store it in a .env file at the project root as
# ALPHAVANTAGE_API_KEY=your_key; it was loaded by load_dotenv above).
ALPHA_VANTAGE_KEY = os.environ.get("ALPHAVANTAGE_API_KEY")
assert ALPHA_VANTAGE_KEY, "Set ALPHAVANTAGE_API_KEY (e.g. in a .env file)."

raw_cache = RAW_DIR / "news_alphavantage.parquet"
progress_dir = RAW_DIR / "av_progress"

# Tickers to force-refetch (e.g. after fixing an alias or a truncated pull).
# Leave empty once the completeness check below is clean.
REFETCH = []

if raw_cache.exists() and not REFETCH:
    news_raw = pd.read_parquet(raw_cache)
    print(f"Loaded {len(news_raw):,} cached Alpha Vantage articles")
else:
    news_raw = fetch_alphavantage(
        UNIVERSE.tickers, ALPHA_VANTAGE_KEY,
        start=ARCHIVE_START,          # 2022-03; archive floor
        end="20251231T0000",
        max_pages=40,                 # paid 75/min tier: go deep
        progress_dir=progress_dir,
        refetch=REFETCH,
    )
    news_raw.to_parquet(raw_cache, index=False)

print(f"Raw articles: {len(news_raw):,}")
print(f"Date range: {news_raw['published_at'].min()} to {news_raw['published_at'].max()}")

Loaded 100,678 cached Alpha Vantage articles
Raw articles: 100,678
Date range: 2022-03-01 07:00:00 to 2025-12-30 23:35:44


In [8]:
# Completeness check: every ticker should span ~2022-03 to 2025-12.
# Any flagged ticker (short coverage, a round-1000 count, or missing) should
# be added to REFETCH above and re-run before trusting the dataset.
report = completeness_report(news_raw, UNIVERSE.tickers)
pd.set_option("display.max_rows", 40)
print(report.to_string())
if report.attrs["missing"]:
    print("\nMISSING entirely:", report.attrs["missing"])
flagged = report.index[report["suspect"]].tolist()
print("\nSuspect tickers:", flagged or "none — dataset looks complete")

            n            earliest              latest  reaches_end  round_1000  suspect
ticker                                                                                 
HD       1679 2022-03-04 13:41:31 2025-12-30 22:08:41         True       False    False
V        1730 2022-03-02 13:25:04 2025-12-30 23:09:23         True       False    False
PG       1785 2022-03-02 13:57:47 2025-12-30 21:09:07         True       False    False
UNH      1855 2022-03-01 16:27:29 2025-12-30 20:08:52         True       False    False
DIS      2193 2022-03-01 16:44:00 2025-12-30 21:07:48         True       False    False
NFLX     2866 2022-03-02 13:52:30 2025-12-30 22:09:00         True       False    False
META     2894 2022-03-01 08:41:00 2025-12-30 21:52:09         True       False    False
GOOGL    3158 2022-03-01 13:40:18 2025-12-30 23:00:00         True       False    False
PFE      3304 2022-03-01 13:48:08 2025-12-30 23:24:57         True       False    False
BA       3497 2022-03-01 13:23:2

In [9]:
# Deduplicate, length-filter, and map every article to its point-in-time
# effective date. This produces the single canonical cleaned news file that
# Notebook 03 scores with FinBERT.
trading_days = pd.DatetimeIndex(prices["date"].drop_duplicates().sort_values())
news = clean_news(news_raw, trading_days)
news.to_parquet(PROCESSED_DIR / "news_clean.parquet", index=False)

print(f"After dedup/length filter: {len(news):,} articles")
news[["published_at", "effective_date", "ticker", "headline",
      "av_sentiment_score"]].head(8)

After dedup/length filter: 95,758 articles


,published_at,effective_date,ticker,headline,av_sentiment_score
0,2022-03-01 13:19:16,2022-03-01,AAPL,Apple Halts Product Sales in Russia After Ukra...,-0.315839
1,2022-03-01 17:24:00,2022-03-01,AAPL,"Google, Meta face penalties in Russia as deadl...",0.247100
2,2022-03-01 07:00:00,2022-03-01,AMZN,Federal Realty Announces Major Overhaul of Hun...,0.338938
3,2022-03-01 13:42:22,2022-03-01,AMZN,File your taxes on time with H&R Block Deluxe ...,0.145472
4,2022-03-01 18:36:56,2022-03-01,AMZN,"At Snap Inc., Making Augmented Reality Work fo...",0.119276
5,2022-03-01 13:23:22,2022-03-01,BA,Boeing and Ford suspend operations in Russia.,-0.419441
6,2022-03-01 16:44:00,2022-03-01,DIS,Comcast Corp. Cl A stock outperforms competito...,-0.234900
7,2022-03-01 13:40:18,2022-03-01,GOOGL,How Does Google Make Money?,0.537457


### 3.2 Auxiliary dataset: Financial PhraseBank

The Financial PhraseBank (Malo et al., 2014) contains approximately 4,800 sentences drawn from financial news, each labeled *positive*, *neutral*, or *negative* by annotators with finance backgrounds, at four inter-annotator agreement levels. We use the 75%-agreement subset, trading a modest reduction in size for substantially cleaner labels, and split it 80/10/10 (train/validation/test) with stratification by class. This corpus serves two roles in Notebook 03: fine-tuning the transformer sentiment model and providing a held-out benchmark of its classification quality before it is trusted to score live headlines.


In [10]:
from src.data.phrasebank import load_phrasebank

splits = load_phrasebank(seed=RANDOM_SEED)
for name, df in splits.items():
    print(f"{name:>5}: {len(df):>5} sentences | "
          f"class balance: {df['label'].value_counts(normalize=True).round(2).to_dict()}")

c:\Users\SyedM\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-08-09 20:44:04,598 HTTP Request: HEAD https://huggingface.co/datasets/takala/financial_phrasebank/resolve/main/data/FinancialPhraseBank-v1.0.zip "HTTP/1.1 302 Found"


train:  2762 sentences | class balance: {1: 0.62, 2: 0.26, 0: 0.12}
  val:   345 sentences | class balance: {1: 0.62, 2: 0.26, 0: 0.12}
 test:   346 sentences | class balance: {1: 0.62, 2: 0.26, 0: 0.12}


## 4. Summary and Hand-off

At the end of this notebook, three clean artifacts exist under `data/processed/`: the cleaned price panel (`prices_clean.parquet`), the cleaned and point-in-time-aligned news table (`news_clean.parquet`), and the trading-day calendar used for alignment.

**What was produced.** The price panel holds 30,140 rows: 20 tickers across 1,507 shared trading sessions from January 2020 through December 2025, with no missing sessions against the union calendar and no bad-print outliers flagged. The news table holds 95,758 articles spanning March 2022 through December 2025, reduced from 100,678 raw articles by removing exact-duplicate wire copies and sub-ten-character fragments. Every article carries its point-in-time `effective_date` (the first session it could have been traded on) alongside both the raw Alpha Vantage sentiment score and the fields needed for FinBERT scoring in Notebook 03.

**Data-quality issues resolved here.** Two problems surfaced and were fixed during ingestion. Alphabet was tagged by the vendor under the symbol GOOG rather than GOOGL, which initially returned zero articles for that name until the symbol mapping was corrected. Separately, a pagination bug silently truncated several heavily-covered tickers at exactly one thousand articles; a completeness check that flags any ticker not spanning the full window (or landing on a suspiciously round count) was added, after which all twenty tickers were confirmed complete.

**Hand-off.** Notebook 03 scores `news_clean.parquet` with FinBERT to produce `news_scored.parquet`; Notebooks 02, 04, 05, 06, and 07 consume the scored file. Because every downstream stage inherits the point-in-time alignment applied here, no later stage can accidentally use information that was unavailable at trading time.

---
### Acknowledgment of AI Tool Use

Portions of the code scaffolding and prose in this notebook were drafted with the assistance of Anthropic's Claude (Anthropic, 2026) and subsequently reviewed, tested, and revised by the author, who takes full responsibility for the final content, design decisions, and results. This acknowledgment is provided in accordance with University of San Diego academic integrity guidelines on the use of generative AI tools.

Anthropic. (2026). *Claude* [Large language model]. https://claude.ai

### References

Aroussi, R. (2023). *yfinance* [Computer software]. https://github.com/ranaroussi/yfinance

Bailey, D. H., Borwein, J. M., López de Prado, M., & Zhu, Q. J. (2014). Pseudo-mathematics and financial charlatanism: The effects of backtest overfitting on out-of-sample performance. *Notices of the American Mathematical Society, 61*(5), 458–471.

Fama, E. F. (1970). Efficient capital markets: A review of theory and empirical work. *The Journal of Finance, 25*(2), 383–417.

Heston, S. L., & Sinha, N. R. (2017). News vs. sentiment: Predicting stock returns from news stories. *Financial Analysts Journal, 73*(3), 67–83.

López de Prado, M. (2018). *Advances in financial machine learning.* Wiley.

Malo, P., Sinha, A., Korhonen, P., Wallenius, J., & Takala, P. (2014). Good debt or bad debt: Detecting semantic orientations in economic texts. *Journal of the Association for Information Science and Technology, 65*(4), 782–796.

Tetlock, P. C. (2007). Giving content to investor sentiment: The role of media in the stock market. *The Journal of Finance, 62*(3), 1139–1168.
